# Bays (2014) Figure 1 d,e,f — GP-Based Equivalent
## Variance, Kurtosis, and Power-Law Exponent Over (λ, γ) Space

### Intellectual Foundation

Bays (2014) showed that a population coding model with divisive normalisation produces
characteristic error distributions whose shape depends on two parameters: tuning width (ω)
and population gain (γ). This figure maps the landscape of these errors.

**Panel d — Variance:** How noisy are recalled orientations? Low gain or broad tuning → high variance.

**Panel e — Kurtosis:** How non-Gaussian are the errors? At low gain, errors develop heavy tails
(positive kurtosis) — the model's signature prediction. This is NOT a mixture of "remembered" and
"guessed" items (as the slot model claims) but a continuous family of distributions.

**Panel f — Exponent:** The power-law relationship V(γ) ∝ γ^α. When α ≈ -1, halving γ doubles
variance (linear regime). When α < -1, the penalty is supralinear (DN regime).

### Key Design Choice: Full Poisson ML Decoder
For GP tuning curves the population does NOT tile uniformly, so Σ_i g_i(θ) fluctuates
across θ. We use the FULL Poisson log-likelihood (retaining the rate-penalty term):
θ̂ = argmax_θ Σ_i [n_i · log g_i(θ) − g_i(θ) · T_d]

### What to play with
- `N_GRID`: Resolution of the (λ, γ) grid (25×25 = 625 points × n_trials each — slow but rich)
- `N_TRIALS`: More trials → smoother grids (10,000 is the paper's standard)
- `LAMBDA_RANGE` and `GAMMA_RANGE`: Zoom in on interesting regions

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import time

np.random.seed(42)

# --- PARAMETERS (MODIFY THESE) ---
M = 100                        # Neurons per population
N_THETA = 64                   # Orientation bins
N_TRIALS = 2000                # Trials per grid point (10k in paper; 2k for speed)
T_D = 0.1                      # Decoding window (s)
SIGMA_SQ = 1e-6                # DN semi-saturation
N_GRID = 12                    # Grid resolution per axis (25 in paper; 12 for speed)
LAMBDA_RANGE = (0.1, 2.5)     # Lengthscale range
GAMMA_RANGE = (1.0, 256.0)    # Gain range (Hz)
SEED = 42

lambdas = np.logspace(np.log2(LAMBDA_RANGE[0]), np.log2(LAMBDA_RANGE[1]), N_GRID, base=2)
gammas = np.logspace(np.log2(GAMMA_RANGE[0]), np.log2(GAMMA_RANGE[1]), N_GRID, base=2)
print(f"Grid: {N_GRID}×{N_GRID} = {N_GRID**2} points, {N_TRIALS} trials each")
print(f"Total trials: {N_GRID**2 * N_TRIALS * 2:,.0f} (×2 for exponent computation)")

In [ ]:
# ============================================================
# SELF-CONTAINED CORE: GP Population + DN + Poisson + ML Decoder
# ============================================================
# Replaces imports from core.encoder.*, core.decoder.*, bays_utils

def periodic_rbf_kernel(thetas, lengthscale):
    """Periodic squared-exponential kernel."""
    diff = thetas[:, None] - thetas[None, :]
    circ = np.abs(np.angle(np.exp(1j * diff)))
    return np.exp(-0.5 * (circ / lengthscale)**2)

def sample_gp_function(K, rng):
    """Sample one function from GP(0, K)."""
    L = np.linalg.cholesky(K + 1e-6 * np.eye(len(K)))
    return L @ rng.randn(len(K))

def generate_population(M, n_theta, lengthscale, n_locations=1, seed=42):
    """Generate M neurons with GP tuning at n_locations. Returns (thetas, f_all)."""
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
    K = periodic_rbf_kernel(thetas, lengthscale)
    f_all = []
    for loc in range(n_locations):
        f_loc = np.zeros((M, n_theta))
        for i in range(M):
            f_loc[i] = sample_gp_function(K, rng)
        f_all.append(f_loc)
    return thetas, f_all

def dn_pointwise(r_pre, gamma, sigma_sq):
    """DN: r^post_i = γ · r^pre_i / (σ² + mean(r^pre))."""
    D = sigma_sq + np.mean(r_pre)
    return gamma * r_pre / D

def generate_spikes(rates, T_d, rng):
    """Poisson spike counts."""
    return rng.poisson(np.maximum(rates, 0) * T_d)

def compute_log_likelihood(counts, g, T_d):
    """Full Poisson ML: LL(θ) = Σ_i [n_i·log g_i(θ) - g_i(θ)·T_d]."""
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    """Signed circular error in [-π, π)."""
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    """Circular variance: 1 - |mean(exp(iε))|."""
    return 1.0 - np.abs(np.mean(np.exp(1j * errors)))

def circular_kurtosis(errors):
    """Circular kurtosis: κ₂/V² where κ₂ = 1-|ρ₂|, V = circ var."""
    V = circular_variance(errors)
    rho2 = np.abs(np.mean(np.exp(2j * errors)))
    kappa2 = 1.0 - rho2
    return kappa2 / max(V**2, 1e-15) if V > 1e-10 else 0.0

def compute_deviation_from_normal(errors, n_bins=50):
    """Deviation of empirical dist from matched circular normal."""
    from scipy.stats import vonmises
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width = bin_edges[1] - bin_edges[0]
    emp, _ = np.histogram(errors, bins=bin_edges, density=True)
    V = circular_variance(errors)
    kappa_fit = max(0.01, 1.0 / V - 1) if V > 0.01 else 100.0
    vm_pdf = vonmises.pdf(centers, kappa_fit)
    return {'bin_centers': centers, 'empirical': emp, 'normal_fit': vm_pdf,
            'deviation': emp - vm_pdf}

# Factorised multi-location decoder
from scipy.special import logsumexp

def compute_spike_weighted_log_tuning(counts, f_list):
    """L_k(θ) = Σ_i n_i · f_{i,k}(θ) for each location k."""
    return [counts @ f_k for f_k in f_list]

def compute_marginal_log_likelihood_efficient(L_list, cued_idx):
    """Efficient factorised ML: L_c(θ) + Σ_{k≠c} logsumexp(L_k)."""
    ll = L_list[cued_idx].copy()
    for k in range(len(L_list)):
        if k != cued_idx:
            ll = ll + logsumexp(L_list[k])
    return ll

print("Core model loaded (GP + DN + Poisson + ML decoder)")

---
## Experiment: Sweep (λ, γ) Grid

### What is occurring
For each (λ, γ) pair:
1. Generate M neurons with GP tuning (lengthscale = λ)
2. Run n_trials: sample θ_true → DN at gain γ → Poisson spikes → Full ML decode → record error
3. Compute circular variance and kurtosis from errors
4. Repeat at γ/2 to get the exponent: α = log₂(V(γ)/V(γ/2))

### Why the exponent matters
α measures how steeply variance grows when gain is halved (equivalent to doubling set size
under DN). α = -1 means linear scaling; α < -1 means supralinear — the hallmark of DN's
compressive nonlinearity.

In [ ]:
# ============================================================
# MAIN GRID SWEEP
# ============================================================
t0 = time.time()

variance_grid = np.full((N_GRID, N_GRID), np.nan)
kurtosis_grid = np.full((N_GRID, N_GRID), np.nan)
variance_half = np.full((N_GRID, N_GRID), np.nan)

for i, lam in enumerate(lambdas):
    thetas, f_all = generate_population(M, N_THETA, lam, n_locations=1, seed=SEED + i*1000)
    g = np.exp(f_all[0])  # Driving inputs

    for j, gam in enumerate(gammas):
        rng = np.random.RandomState(SEED + i*N_GRID + j)
        errors = np.empty(N_TRIALS)
        for t in range(N_TRIALS):
            idx_true = rng.randint(N_THETA)
            rates = dn_pointwise(g[:, idx_true], gam, SIGMA_SQ)
            counts = generate_spikes(rates, T_D, rng)
            ll = compute_log_likelihood(counts, g, T_D)
            errors[t] = compute_circular_error(thetas[idx_true], thetas[np.argmax(ll)])

        variance_grid[j, i] = circular_variance(errors)
        kurtosis_grid[j, i] = circular_kurtosis(errors)

        # Repeat at γ/2 for exponent
        rng2 = np.random.RandomState(SEED + i*N_GRID + j + N_GRID**2)
        errors_half = np.empty(N_TRIALS)
        for t in range(N_TRIALS):
            idx_true = rng2.randint(N_THETA)
            rates = dn_pointwise(g[:, idx_true], gam/2, SIGMA_SQ)
            counts = generate_spikes(rates, T_D, rng2)
            ll = compute_log_likelihood(counts, g, T_D)
            errors_half[t] = compute_circular_error(thetas[idx_true], thetas[np.argmax(ll)])
        variance_half[j, i] = circular_variance(errors_half)

    pct = (i+1)/N_GRID*100
    print(f"  λ={lam:.3f} done ({pct:.0f}%, {time.time()-t0:.0f}s)")

# Exponent: α = log₂(V(γ)/V(γ/2))
with np.errstate(divide='ignore', invalid='ignore'):
    ratio = np.where(variance_half > 1e-15, variance_grid / variance_half, np.nan)
    exponent_grid = np.where(ratio > 0, np.log2(ratio), np.nan)

print(f"\nDone in {time.time()-t0:.1f}s")

---
## Plots: Panels d, e, f

### Panel d (Variance)
Blue = low variance (good recall); red = high variance. The "yellow band" corresponds
to human-like performance.

### Panel e (Kurtosis)
High kurtosis (red) at low gain = heavy tails = the population coding model's signature.
This is what distinguishes it from slot models that predict Gaussian + uniform mixtures.

### Panel f (Exponent)
α ≈ 0 (red) means variance is insensitive to gain changes. α ≈ -1 means linear.
α < -1 (blue) means supralinear — the regime where DN dominates.

In [ ]:
# ============================================================
# THREE-PANEL FIGURE
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
extent = [np.log2(lambdas[0]), np.log2(lambdas[-1]),
          np.log2(gammas[0]), np.log2(gammas[-1])]

def fmt_ax(ax, label):
    ax.set_xlabel(r'lengthscale $\lambda$')
    ax.set_ylabel(r'gain $\gamma$ (Hz)')
    xt = np.array([0.125, 0.25, 0.5, 1.0, 2.0])
    xt = xt[(xt >= lambdas[0]*0.9) & (xt <= lambdas[-1]*1.1)]
    ax.set_xticks(np.log2(xt)); ax.set_xticklabels([f'{v:g}' for v in xt])
    yt = np.array([1, 4, 16, 64, 256])
    yt = yt[(yt >= gammas[0]*0.9) & (yt <= gammas[-1]*1.1)]
    ax.set_yticks(np.log2(yt)); ax.set_yticklabels([f'{v:g}' for v in yt])
    ax.text(-0.1, 1.06, f'$\\mathbf{{{label}}}$', transform=ax.transAxes,
            fontsize=15, fontweight='bold', va='top')

im0 = axes[0].imshow(np.clip(variance_grid, 1e-3, 10), origin='lower', aspect='auto',
    extent=extent, norm=mcolors.LogNorm(vmin=0.001, vmax=10), cmap='jet', interpolation='bilinear')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04).set_label('variance')
fmt_ax(axes[0], 'd')

im1 = axes[1].imshow(np.clip(kurtosis_grid, 0.01, 100), origin='lower', aspect='auto',
    extent=extent, norm=mcolors.LogNorm(vmin=0.01, vmax=100), cmap='jet', interpolation='bilinear')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04).set_label('kurtosis')
fmt_ax(axes[1], 'e')

im2 = axes[2].imshow(np.clip(exponent_grid, -3, 0), origin='lower', aspect='auto',
    extent=extent, vmin=-3, vmax=0, cmap='jet', interpolation='bilinear')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04).set_label('exponent')
fmt_ax(axes[2], 'f')

fig.suptitle('GP Population Coding — Bays (2014) Fig 1 d,e,f Equivalent', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()